## 2 Pizzas

- Harina 500 gr
- condimento pizza 3 gr
- sal 10 gr
- pimenton 2 gr
- Oregano 5 gr
- 1 lata tomate
- 1 sobre levadura
- Aji molido 2 gr
- 1/6 ajo
- Queso mozzarela 600 gr


In [0]:
WITH receta_pizza(
  SELECT 
  nombre,
  precio_normalizado AS precio,
  CAST(CASE 
    WHEN nombre in ("Harina Trigo 0000 MORIXE 1kg","Queso Mozzarella BARRAZA 1kg") then '1000'
    ELSE NULLIF(REGEXP_EXTRACT(trim(nombre), '(\\d+)', 0),'') 
  END AS INT) AS peso_en_gramos,

  CAST(CASE 
    WHEN nombre = "Ajo" THEN '1'
    ELSE NULL
  END AS INT) AS unidades,
  fecha_extraccion,
  url
  FROM supermercadoetl_prod.silver.precios_productos
  WHERE nombre IN ("Harina Trigo 0000 MORIXE 1kg",
                  "Condimento Para Pizza La Campagnola Sob 23 Grm",
                  "Sal Fina CELUSAL 500g",
                  "Pimenton Seleccionado Dos Anclas Pou 50 Grm",
                  "Orégano Dos Anclas 25 Grm",
                  "Tomate Perita ARCOR Lata 400 Gr",
                  "Levadura Seca CALSA Mi Pan 10 Gr",
                  "Aji Molido DOS ANCLAS Sobre 50 Gr",
                  "Ajo",
                  "Queso Mozzarella BARRAZA 1kg")
),
precios_unitarios AS (
  SELECT 
  nombre,
  precio,
  peso_en_gramos,
  ROUND(precio / peso_en_gramos,2) AS precio_por_gramo,
  CASE 
    WHEN nombre = "Ajo" THEN ROUND(precio * unidades)
    ELSE NULL
  END AS precio_por_unidad,
  unidades,
  fecha_extraccion,
  url
  FROM receta_pizza
),
cantidades_receta_pizza AS (
  SELECT * FROM VALUES
    ('Harina Trigo 0000 MORIXE 1kg', 500),
    ('Condimento Para Pizza La Campagnola Sob 23 Grm', 3),
    ('Sal Fina CELUSAL 500g', 10),
    ('Pimenton Seleccionado Dos Anclas Pou 50 Grm', 2),
    ('Orégano Dos Anclas 25 Grm', 5),
    ('Tomate Perita ARCOR Lata 400 Gr', 400),
    ('Levadura Seca CALSA Mi Pan 10 Gr', 10),
    ('Aji Molido DOS ANCLAS Sobre 50 Gr',2),
    ('Ajo', 1.0/6.0),
    ('Queso Mozzarella BARRAZA 1kg', 600)
  AS t(nombre, cantidad_receta)
),
costo_pizza AS(

SELECT 
  p.nombre,
  p.precio,
  COALESCE(precio_por_gramo, precio_por_unidad) AS precio_unitario,
  c.cantidad_receta,
  ROUND(COALESCE(precio_por_gramo, precio_por_unidad) * c.cantidad_receta,2) AS costo,
  p.fecha_extraccion,
  p.url
FROM precios_unitarios p
LEFT JOIN cantidades_receta_pizza c
ON TRIM(LOWER(p.nombre))  = TRIM(LOWER(c.nombre)) 
)
SELECT 
fecha_extraccion,
ROUND(SUM(costo) / 2,2) AS costo_pizza 
FROM costo_pizza
GROUP BY fecha_extraccion
HAVING fecha_extraccion >=  '2026-04-15'
ORDER BY fecha_extraccion ASC

## EMPANADAS
- 1kg carne picada
- 3 paquetes de tapas de empanadas x12
- 2 atados puerro
- 2 atados cebolla de verdeo
- 1 cebolla
- 2 zanahorias
- 1 morron
- sal
- condimento empanadas
- 1 paquete de aceitunas (300 gr)
- 5 huevos

In [0]:
WITH cantidades_receta_empanadas AS (
  SELECT * FROM VALUES
    ('Picada Desgrasada',1000),
      ('Tapa Para Empanadas X12 SIGNO DE ORO 500g',3),
      ('Cebolla',200),
      ('Puerro',2),
      ('Cebolla de verdeo',2),
      ('Zanahoria',150),
      ('Morron rojo',250),
      ('Sal Fina CELUSAL 500g',50),
      ('Condimento Para Empanadas La Parmesana 15g',5),
      ('Aceitunas Verdes Descarozadas Morixe 300g',1),
      ('Huevo Blanco Maple X 30u',5)
  AS t(nombre, cantidad_receta)
)

SELECT * FROM cantidades_receta_empanadas

In [0]:
WITH receta_empanadas(
  SELECT 
  nombre,
  precio_normalizado AS precio,
  CAST(CASE 
    WHEN nombre in ('Picada Desgrasada',
                    'Zanahoria',
                    'Morron rojo',
                    'Cebolla') then '1000'
    WHEN nombre in ('Aceitunas Verdes Descarozadas Morixe 300g',
                    'Huevo Blanco Maple X 30u',
                    'Tapa Para Empanadas X12 SIGNO DE ORO 500g'
                    ) THEN NULL
    ELSE NULLIF(REGEXP_EXTRACT(trim(nombre), '(\\d+)', 0),'') 
  END AS INT) AS peso_en_gramos,
  CAST(CASE 
    WHEN nombre in ('Puerro',
                    'Cebolla de verdeo',                    
                    'Tapa Para Empanadas X12 SIGNO DE ORO 500g',
                    'Aceitunas Verdes Descarozadas Morixe 300g') THEN '1'
    WHEN nombre = 'Huevo Blanco Maple X 30u' THEN '30'
    ELSE NULL
  END AS INT) AS unidades,
  fecha_extraccion,
  url
  FROM supermercadoetl_prod.silver.precios_productos
  WHERE nombre IN       ('Picada Desgrasada',
                          'Tapa Para Empanadas X12 SIGNO DE ORO 500g',
                          'Cebolla',
                          'Puerro',
                          'Cebolla de verdeo',
                          'Zanahoria',
                          'Morron rojo',
                          'Sal Fina CELUSAL 500g',
                          'Condimento Para Empanadas La Parmesana 15g',
                          'Aceitunas Verdes Descarozadas Morixe 300g',
                          'Huevo Blanco Maple X 30u')
                    ),
precios_unitarios AS (
  SELECT 
  nombre,
  precio,
  peso_en_gramos,
  ROUND(precio / peso_en_gramos,2) AS precio_por_gramo,
  CASE 
    WHEN nombre in ('Tapa Para Empanadas X12 SIGNO DE ORO 500g',
                    'Aceitunas Verdes Descarozadas Morixe 300g',
                    'Puerro',
                    'Cebolla de verdeo'
                    ) THEN ROUND(precio * unidades)
    WHEN nombre = "Huevo Blanco Maple X 30u" THEN ROUND(precio / unidades)
    ELSE NULL
  END AS precio_por_unidad,
  unidades,
  fecha_extraccion,
  url
  FROM receta_empanadas
),
cantidades_receta_empanadas AS (
  SELECT * FROM VALUES
    ('Picada Desgrasada',1000),
      ('Tapa Para Empanadas X12 SIGNO DE ORO 500g',3),
      ('Cebolla',200),
      ('Puerro',2),
      ('Cebolla de verdeo',2),
      ('Zanahoria',150),
      ('Morron rojo',250),
      ('Sal Fina CELUSAL 500g',50),
      ('Condimento Para Empanadas La Parmesana 15g',5),
      ('Aceitunas Verdes Descarozadas Morixe 300g',1),
      ('Huevo Blanco Maple X 30u',5)
  AS t(nombre, cantidad_receta)
),
costo_empanadas AS(
  SELECT 
    p.nombre,
    p.precio,
    COALESCE(precio_por_gramo, precio_por_unidad) AS precio_unitario,
    c.cantidad_receta,
    ROUND(COALESCE(precio_por_gramo, precio_por_unidad) * c.cantidad_receta,2) AS costo,
    p.fecha_extraccion,
    p.url
  FROM precios_unitarios p
  LEFT JOIN cantidades_receta_empanadas c
  ON TRIM(LOWER(p.nombre))  = TRIM(LOWER(c.nombre)) 
)

SELECT 
fecha_extraccion,
ROUND(SUM(costo)/3,2) AS costo_docena_empanadas 
FROM costo_empanadas
GROUP BY fecha_extraccion
HAVING fecha_extraccion >=  '2026-04-15'
ORDER BY fecha_extraccion ASC



-- SELECT 
-- *
-- FROM costo_empanadas
-- ORDER BY nombre,fecha_extraccion asc


In [0]:
-- Query unificada que calcula el costo de todas las recetas en una sola consulta
WITH todos_productos AS (
  SELECT 
    nombre,
    precio_normalizado AS precio,
    CAST(CASE 
      -- Productos con peso en kg -> convertir a gramos
      WHEN nombre IN ('Harina Trigo 0000 MORIXE 1kg','Queso Mozzarella BARRAZA 1kg',
                      'Picada Desgrasada','Zanahoria','Morron rojo','Cebolla') THEN 1000
      -- Extraer gramos del nombre para productos con peso
      WHEN nombre LIKE '%Grm%' OR (nombre LIKE '%g%' AND nombre NOT LIKE '%X%') THEN 
        CAST(NULLIF(REGEXP_EXTRACT(TRIM(nombre), '(\\d+)', 0),'') AS INT)
      -- Productos sin peso (solo unidad)
      WHEN nombre IN ('Ajo','Puerro','Cebolla de verdeo',
                      'Tomate Perita ARCOR Lata 400 Gr',
                      'Levadura Seca CALSA Mi Pan 10 Gr') THEN NULL
      -- Aceitunas: extraer 300 del nombre (paquete de 300g)
      WHEN nombre = 'Aceitunas Verdes Descarozadas Morixe 300g' THEN 300
      ELSE NULL
    END AS INT) AS peso_en_gramos,
    CAST(CASE 
      -- Ajo: 1 unidad
      WHEN nombre = 'Ajo' THEN 1
      -- Puerro y cebolla de verdeo: 1 unidad por atado
      WHEN nombre IN ('Puerro','Cebolla de verdeo') THEN 1
      -- Aceitunas: 1 paquete
      WHEN nombre = 'Aceitunas Verdes Descarozadas Morixe 300g' THEN 1
      -- Tapas: extraer 12 del nombre (paquete contiene 12 tapas)
      WHEN nombre = 'Tapa Para Empanadas X12 SIGNO DE ORO 500g' THEN 12
      -- Huevos: extraer 30 del nombre (maple contiene 30 huevos)
      WHEN nombre = 'Huevo Blanco Maple X 30u' THEN 30
      -- Tomate y levadura: 1 unidad
      WHEN nombre IN ('Tomate Perita ARCOR Lata 400 Gr',
                      'Levadura Seca CALSA Mi Pan 10 Gr') THEN 1
      ELSE NULL
    END AS INT) AS unidades,
    fecha_extraccion,
    url
  FROM supermercadoetl_prod.silver.precios_productos
  WHERE nombre IN (
    -- Ingredientes Pizza
    'Harina Trigo 0000 MORIXE 1kg',
    'Condimento Para Pizza La Campagnola Sob 23 Grm',
    'Sal Fina CELUSAL 500g',
    'Pimenton Seleccionado Dos Anclas Pou 50 Grm',
    'Orégano Dos Anclas 25 Grm',
    'Tomate Perita ARCOR Lata 400 Gr',
    'Levadura Seca CALSA Mi Pan 10 Gr',
    'Aji Molido DOS ANCLAS Sobre 50 Gr',
    'Ajo',
    'Queso Mozzarella BARRAZA 1kg',
    -- Ingredientes Empanadas
    'Picada Desgrasada',
    'Tapa Para Empanadas X12 SIGNO DE ORO 500g',
    'Cebolla',
    'Puerro',
    'Cebolla de verdeo',
    'Zanahoria',
    'Morron rojo',
    'Condimento Para Empanadas La Parmesana 15g',
    'Aceitunas Verdes Descarozadas Morixe 300g',
    'Huevo Blanco Maple X 30u'
  )
),
precios_unitarios AS (
  SELECT 
    nombre,
    precio,
    peso_en_gramos,
    ROUND(precio / peso_en_gramos, 2) AS precio_por_gramo,
    CASE 
      -- Tapas X12: dividir precio del paquete por 12 tapas
      WHEN nombre = 'Tapa Para Empanadas X12 SIGNO DE ORO 500g' THEN ROUND(precio / unidades, 2)
      -- Aceitunas 300g: dividir precio del paquete por 300 gramos
      WHEN nombre = 'Aceitunas Verdes Descarozadas Morixe 300g' THEN ROUND(precio / 300, 4)
      -- Huevos X30u: dividir precio del maple por 30 huevos
      WHEN nombre = 'Huevo Blanco Maple X 30u' THEN ROUND(precio / unidades, 2)
      -- Productos de unidad simple (puerro, cebolla verdeo, ajo, tomate, levadura)
      WHEN nombre IN ('Puerro','Cebolla de verdeo','Ajo',
                      'Tomate Perita ARCOR Lata 400 Gr',
                      'Levadura Seca CALSA Mi Pan 10 Gr') THEN ROUND(precio / COALESCE(unidades, 1), 2)
      ELSE NULL
    END AS precio_por_unidad,
    unidades,
    fecha_extraccion,
    url
  FROM todos_productos
),
cantidades_recetas AS (
  SELECT * FROM VALUES
    -- Pizza (2 pizzas)
    ('Pizza', 'Harina Trigo 0000 MORIXE 1kg', 500),
    ('Pizza', 'Condimento Para Pizza La Campagnola Sob 23 Grm', 3),
    ('Pizza', 'Sal Fina CELUSAL 500g', 10),
    ('Pizza', 'Pimenton Seleccionado Dos Anclas Pou 50 Grm', 2),
    ('Pizza', 'Orégano Dos Anclas 25 Grm', 5),
    ('Pizza', 'Tomate Perita ARCOR Lata 400 Gr', 1),
    ('Pizza', 'Levadura Seca CALSA Mi Pan 10 Gr', 1),
    ('Pizza', 'Aji Molido DOS ANCLAS Sobre 50 Gr', 2),
    ('Pizza', 'Ajo', 1.0/6.0),
    ('Pizza', 'Queso Mozzarella BARRAZA 1kg', 600),
    -- Empanadas (3 docenas = 36 empanadas)
    ('Empanadas', 'Picada Desgrasada', 1000),
    ('Empanadas', 'Tapa Para Empanadas X12 SIGNO DE ORO 500g', 3),
    ('Empanadas', 'Cebolla', 200),
    ('Empanadas', 'Puerro', 2),
    ('Empanadas', 'Cebolla de verdeo', 2),
    ('Empanadas', 'Zanahoria', 150),
    ('Empanadas', 'Morron rojo', 250),
    ('Empanadas', 'Sal Fina CELUSAL 500g', 50),
    ('Empanadas', 'Condimento Para Empanadas La Parmesana 15g', 5),
    ('Empanadas', 'Aceitunas Verdes Descarozadas Morixe 300g', 1),
    ('Empanadas', 'Huevo Blanco Maple X 30u', 5)
  AS t(receta, nombre, cantidad_receta)
),
costos_por_receta AS (
  SELECT 
    c.receta,
    p.nombre,
    p.precio,
    COALESCE(p.precio_por_gramo, p.precio_por_unidad) AS precio_unitario,
    c.cantidad_receta,
    ROUND(COALESCE(p.precio_por_gramo, p.precio_por_unidad) * c.cantidad_receta, 2) AS costo,
    p.fecha_extraccion,
    p.url
  FROM precios_unitarios p
  INNER JOIN cantidades_recetas c
    ON TRIM(LOWER(p.nombre)) = TRIM(LOWER(c.nombre))
)
SELECT 
  fecha_extraccion,
  ROUND(SUM(CASE WHEN receta = 'Pizza' THEN costo ELSE 0 END) / 2, 2) AS costo_pizza,
  ROUND(SUM(CASE WHEN receta = 'Empanadas' THEN costo ELSE 0 END) / 3, 2) AS costo_docena_empanadas,
  ROUND(SUM(costo), 2) AS costo_total_recetas
FROM costos_por_receta
GROUP BY fecha_extraccion
HAVING fecha_extraccion >= '2026-04-15'
ORDER BY fecha_extraccion ASC

## Arquitectura Generalizada para Cálculo de Recetas

### 1. Catálogo de Productos (Tabla Base)

Necesitas normalizar todos los productos con estas columnas:

| Campo | Descripción | Ejemplo |
|-------|-------------|----------|
| `nombre` | Nombre del producto | "Morron rojo" |
| `precio` | Precio de venta | 3799.00 |
| `tipo_medida` | peso / volumen / unidad | "peso" |
| `unidad_venta` | Cómo se vende | "kg" / "litro" / "unidad" |
| `cantidad_venta` | Cantidad en unidad venta | 1000 (1kg = 1000g) |
| `precio_por_unidad_base` | Precio normalizado | 3.799 ($/gramo) |

### 2. Conversiones de Unidades

**Peso:**
- 1 kg = 1000 gramos
- Precio base: $/gramo

**Volumen:**
- 1 litro = 1000 ml
- Precio base: $/ml

**Unidad:**
- Productos discretos (huevos, ajos, paquetes)
- Precio base: $/unidad

### 3. Estructura de Recetas

| Campo | Descripción | Ejemplo |
|-------|-------------|----------|
| `receta` | Nombre de la receta | "Pizza" |
| `ingrediente` | Nombre del producto | "Morron rojo" |
| `cantidad` | Cantidad necesaria | 150 |
| `unidad_receta` | Unidad de la cantidad | "gramos" / "ml" / "unidades" |
| `rendimiento` | Cuántas porciones hace | 2 (pizzas) |

### 4. Cálculo del Costo

```sql
Costo_ingrediente = precio_por_unidad_base × cantidad_receta
Costo_por_porcion = SUM(Costo_ingrediente) / rendimiento
```

### 5. Ejemplo Práctico

**Producto:** Morron rojo
- Precio: $3799/kg
- Tipo: peso
- Precio base: $3.799/gramo

**Receta:** Pizza necesita 150g de morron
- Costo: 3.799 × 150 = $569.85

**Producto:** Leche
- Precio: $2500/litro
- Tipo: volumen  
- Precio base: $2.50/ml

**Receta:** Salsa bechamel necesita 500ml de leche
- Costo: 2.50 × 500 = $1250.00

In [0]:
-- Implementación práctica de la matriz de productos
WITH catalogo_productos AS (
  SELECT 
    nombre,
    precio_normalizado AS precio,
    fecha_extraccion,
    
    -- Clasificación de tipo de medida
    CASE 
      WHEN nombre LIKE '%kg%' OR nombre LIKE '%Grm%' OR nombre IN (
        'Picada Desgrasada','Zanahoria','Morron rojo','Cebolla',
        'Harina Trigo 0000 MORIXE 1kg','Queso Mozzarella BARRAZA 1kg'
      ) THEN 'peso'
      WHEN nombre LIKE '%Litro%' OR nombre LIKE '%ml%' THEN 'volumen'
      ELSE 'unidad'
    END AS tipo_medida,
    
    -- Cantidad en la unidad de venta (normalizado a unidad base)
    CAST(CASE 
      -- Productos en kg -> convertir a gramos
      WHEN nombre IN ('Harina Trigo 0000 MORIXE 1kg','Queso Mozzarella BARRAZA 1kg',
                      'Picada Desgrasada','Zanahoria','Morron rojo','Cebolla') THEN 1000
      -- Extraer gramos del nombre
      WHEN nombre LIKE '%Grm%' OR nombre LIKE '%g%' THEN 
        CAST(NULLIF(REGEXP_EXTRACT(TRIM(nombre), '(\\d+)', 0),'') AS INT)
      -- Productos por unidad
      WHEN nombre = 'Huevo Blanco Maple X 30u' THEN 30
      WHEN nombre IN ('Ajo','Puerro','Cebolla de verdeo','Aceitunas Verdes Descarozadas Morixe 300g',
                      'Tapa Para Empanadas X12 SIGNO DE ORO 500g','Tomate Perita ARCOR Lata 400 Gr',
                      'Levadura Seca CALSA Mi Pan 10 Gr') THEN 1
      -- Si tiene litros, convertir a ml (ejemplo)
      WHEN nombre LIKE '%Litro%' THEN 1000
      ELSE NULL
    END AS INT) AS cantidad_venta,
    
    -- Unidad de venta
    CASE 
      WHEN nombre LIKE '%kg%' OR nombre LIKE '%Grm%' OR nombre IN (
        'Picada Desgrasada','Zanahoria','Morron rojo','Cebolla',
        'Harina Trigo 0000 MORIXE 1kg','Queso Mozzarella BARRAZA 1kg'
      ) THEN 'gramos'
      WHEN nombre LIKE '%Litro%' OR nombre LIKE '%ml%' THEN 'ml'
      ELSE 'unidades'
    END AS unidad_venta,
    
    url
  FROM supermercadoetl_prod.silver.precios_productos
  WHERE nombre IN (
    'Harina Trigo 0000 MORIXE 1kg',
    'Condimento Para Pizza La Campagnola Sob 23 Grm',
    'Sal Fina CELUSAL 500g',
    'Pimenton Seleccionado Dos Anclas Pou 50 Grm',
    'Orégano Dos Anclas 25 Grm',
    'Tomate Perita ARCOR Lata 400 Gr',
    'Levadura Seca CALSA Mi Pan 10 Gr',
    'Aji Molido DOS ANCLAS Sobre 50 Gr',
    'Ajo',
    'Queso Mozzarella BARRAZA 1kg',
    'Picada Desgrasada',
    'Tapa Para Empanadas X12 SIGNO DE ORO 500g',
    'Cebolla',
    'Puerro',
    'Cebolla de verdeo',
    'Zanahoria',
    'Morron rojo',
    'Condimento Para Empanadas La Parmesana 15g',
    'Aceitunas Verdes Descarozadas Morixe 300g',
    'Huevo Blanco Maple X 30u'
  )
),
catalogo_normalizado AS (
  SELECT 
    nombre,
    precio,
    tipo_medida,
    unidad_venta,
    cantidad_venta,
    -- Precio por unidad base ($/gramo, $/ml, $/unidad)
    ROUND(precio / cantidad_venta, 4) AS precio_por_unidad_base,
    fecha_extraccion,
    url
  FROM catalogo_productos
),
recetas_definicion AS (
  SELECT * FROM VALUES
    -- Pizza (rinde 2 pizzas)
    ('Pizza', 'Harina Trigo 0000 MORIXE 1kg', 500, 'gramos', 2),
    ('Pizza', 'Condimento Para Pizza La Campagnola Sob 23 Grm', 3, 'gramos', 2),
    ('Pizza', 'Sal Fina CELUSAL 500g', 10, 'gramos', 2),
    ('Pizza', 'Pimenton Seleccionado Dos Anclas Pou 50 Grm', 2, 'gramos', 2),
    ('Pizza', 'Orégano Dos Anclas 25 Grm', 5, 'gramos', 2),
    ('Pizza', 'Tomate Perita ARCOR Lata 400 Gr', 1, 'unidades', 2),
    ('Pizza', 'Levadura Seca CALSA Mi Pan 10 Gr', 1, 'unidades', 2),
    ('Pizza', 'Aji Molido DOS ANCLAS Sobre 50 Gr', 2, 'gramos', 2),
    ('Pizza', 'Ajo', 0.167, 'unidades', 2), -- 1/6 de ajo
    ('Pizza', 'Queso Mozzarella BARRAZA 1kg', 600, 'gramos', 2),
    -- Empanadas (rinde 3 docenas = 36 empanadas)
    ('Empanadas', 'Picada Desgrasada', 1000, 'gramos', 3),
    ('Empanadas', 'Tapa Para Empanadas X12 SIGNO DE ORO 500g', 3, 'unidades', 3),
    ('Empanadas', 'Cebolla', 200, 'gramos', 3),
    ('Empanadas', 'Puerro', 2, 'unidades', 3),
    ('Empanadas', 'Cebolla de verdeo', 2, 'unidades', 3),
    ('Empanadas', 'Zanahoria', 150, 'gramos', 3),
    ('Empanadas', 'Morron rojo', 250, 'gramos', 3),
    ('Empanadas', 'Sal Fina CELUSAL 500g', 50, 'gramos', 3),
    ('Empanadas', 'Condimento Para Empanadas La Parmesana 15g', 5, 'gramos', 3),
    ('Empanadas', 'Aceitunas Verdes Descarozadas Morixe 300g', 1, 'unidades', 3),
    ('Empanadas', 'Huevo Blanco Maple X 30u', 5, 'unidades', 3)
  AS t(receta, ingrediente, cantidad_necesaria, unidad_receta, rendimiento_porciones)
),
recetas_all_ingredientes AS (
  -- Todos los ingredientes definidos en las recetas
  SELECT DISTINCT receta, ingrediente
  FROM recetas_definicion
),
productos_matcheados AS (
  -- Ver qué ingredientes tienen match en la tabla
  SELECT 
    r.receta,
    r.ingrediente,
    c.nombre AS nombre_en_tabla,
    CASE WHEN c.nombre IS NOT NULL THEN 'MATCH' ELSE 'SIN MATCH' END AS estado
  FROM recetas_all_ingredientes r
  LEFT JOIN catalogo_normalizado c
    ON TRIM(LOWER(c.nombre)) = TRIM(LOWER(r.ingrediente))
  WHERE c.fecha_extraccion >= '2026-04-15' OR c.fecha_extraccion IS NULL
),
calculo_costos AS (
  SELECT 
    r.receta,
    r.ingrediente,
    c.tipo_medida,
    c.precio AS precio_producto,
    c.cantidad_venta,
    c.precio_por_unidad_base,
    c.unidad_venta,
    r.cantidad_necesaria,
    r.unidad_receta,
    -- Cálculo del costo del ingrediente
    ROUND(c.precio_por_unidad_base * r.cantidad_necesaria, 2) AS costo_ingrediente,
    r.rendimiento_porciones,
    c.fecha_extraccion
  FROM catalogo_normalizado c
  INNER JOIN recetas_definicion r
    ON TRIM(LOWER(c.nombre)) = TRIM(LOWER(r.ingrediente))
)

-- QUERY DE DIAGNÓSTICO: Ver qué ingredientes NO están matcheando
SELECT * FROM productos_matcheados
WHERE estado = 'SIN MATCH'
ORDER BY receta, ingrediente

/*
-- QUERY ORIGINAL: Resumen por receta
SELECT 
  receta,
  fecha_extraccion,
  ROUND(SUM(costo_ingrediente), 2) AS costo_total,
  MAX(rendimiento_porciones) AS porciones,
  ROUND(SUM(costo_ingrediente) / MAX(rendimiento_porciones), 2) AS costo_por_porcion,
  CASE WHEN receta = "Pizza" then 'Porcion = 1 pizza'
       WHEN receta = "Empanadas" then 'Porcion = Docena'
  ELSE NULL
  END AS comentarios
FROM calculo_costos
WHERE fecha_extraccion >= '2026-04-15'
GROUP BY receta, fecha_extraccion
ORDER BY receta, fecha_extraccion
*/

-- Para ver detalle por ingrediente, descomenta:
-- SELECT * FROM calculo_costos 
-- WHERE fecha_extraccion >= '2026-04-15'
-- ORDER BY receta, ingrediente, fecha_extraccion

## Cómo Agregar Productos de Volumen (Leche, Aceite, etc.)

### Paso 1: Agregar productos líquidos a la tabla base

Ejemplo de productos que medirías en litros/ml:

```sql
-- En el WHERE del CTE catalogo_productos, agregar:
'Leche Entera La Serenisima 1L',
'Aceite de Girasol Cocinero 900ml',
'Crema de Leche La Serenisima 200ml'
```

### Paso 2: El sistema automáticamente clasifica por tipo_medida

El CASE en `catalogo_productos` ya detecta:
- `WHEN nombre LIKE '%Litro%' OR nombre LIKE '%ml%' THEN 'volumen'`
- Convierte a ml como unidad base (1 litro = 1000 ml)
- Calcula `precio_por_unidad_base` en $/ml

### Paso 3: Definir la receta con unidades de volumen

```sql
-- En el CTE recetas_definicion:
('Salsa Bechamel', 'Leche Entera La Serenisima 1L', 500, 'ml', 4),
('Salsa Bechamel', 'Harina Trigo 0000 MORIXE 1kg', 50, 'gramos', 4),
('Salsa Bechamel', 'Sal Fina CELUSAL 500g', 5, 'gramos', 4)
```

### Paso 4: El cálculo se hace automático

```
Costo = precio_por_ml × cantidad_en_ml
```

### Ventajas de este Enfoque

✅ **Unificado**: Mismo sistema para peso, volumen y unidades  
✅ **Escalable**: Agregar nuevas recetas solo requiere añadir filas a `recetas_definicion`  
✅ **Mantenible**: Si cambia el precio, se refleja automáticamente en todas las recetas  
✅ **Flexible**: Puedes calcular costo por porción, por docena, por kg, etc.  

### Siguiente Paso: Crear Tabla Permanente

Para productivizar esto, considera crear:

```sql
CREATE TABLE supermercadoetl_prod.gold.catalogo_productos_normalizado AS
SELECT * FROM catalogo_normalizado;

CREATE TABLE supermercadoetl_prod.gold.recetas_definicion AS  
SELECT * FROM recetas_definicion;
```

Luego puedes calcular cualquier receta con un simple JOIN.